# Project 03: Customer Churn / Sales Trend Analysis

**Track:** Data Science & AI  
**Level:** Intermediate  
**Dataset:** Simulated e-commerce orders (generated with a fixed random seed)

## Business questions
1. Which months generate the highest and lowest revenue?
2. How does revenue change month over month?
3. Which product categories contribute the most revenue?
4. What share of revenue comes from the top 10% of customers?
5. What actions should the business take based on these findings?

> **Data note:** This project uses synthetic, generated order data. It does not contain real customer information. Results describe this simulated dataset only.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


## 1. Generate or load the dataset\nThe generator below follows the sample structure in the learner guide and creates 2,000 orders for 300 possible customers.

In [ ]:
rng = np.random.default_rng(42)
n = 2000

orders = pd.DataFrame({
    "order_date": pd.to_datetime("2025-01-01") +
                  pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
    "customer_id": rng.integers(1, 301, n),
    "category": rng.choice(
        ["Electronics", "Fashion", "Home", "Books"],
        n,
        p=[0.30, 0.35, 0.20, 0.15]
    ),
    "amount": rng.gamma(2.0, 600, n).round(2)
})

display(orders.head())
print("Shape:", orders.shape)
print("\\nData types:")
print(orders.dtypes)
print("\\nMissing values:")
print(orders.isnull().sum())
print("\\nDuplicate rows:", orders.duplicated().sum())


## 2. Data preparation and feature engineering\nConvert the order date and extract month, weekday, and quarter features.

In [ ]:
orders = orders.drop_duplicates().dropna().copy()
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["month"] = orders["order_date"].dt.to_period("M")
orders["weekday"] = orders["order_date"].dt.day_name()
orders["quarter"] = orders["order_date"].dt.to_period("Q").astype(str)

display(orders.head())


## 3. Monthly revenue trend and month-over-month growth

In [ ]:
monthly = orders.groupby("month")["amount"].sum().sort_index()
monthly_growth = monthly.pct_change() * 100

monthly_summary = pd.DataFrame({
    "revenue": monthly,
    "month_over_month_growth_pct": monthly_growth
})
display(monthly_summary.round(2))

best_month = monthly.idxmax()
worst_month = monthly.idxmin()
print("Best month:", best_month, "| Revenue:", round(monthly.max(), 2))
print("Worst month:", worst_month, "| Revenue:", round(monthly.min(), 2))


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly.index.astype(str), monthly.values, marker="o")
plt.scatter([str(best_month)], [monthly.max()], label=f"Best: {best_month}", s=90)
plt.scatter([str(worst_month)], [monthly.min()], label=f"Worst: {worst_month}", s=90)
plt.title("Monthly E-commerce Revenue — 2025")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.bar(monthly_growth.index.astype(str), monthly_growth.fillna(0).values)
plt.axhline(0, linewidth=1)
plt.title("Month-over-Month Revenue Growth")
plt.xlabel("Month")
plt.ylabel("Growth (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Category performance using groupby and pivot_table

In [ ]:
by_category = orders.groupby("category")["amount"].agg(
    total_revenue="sum",
    average_order_value="mean",
    order_count="count"
).sort_values("total_revenue", ascending=False)

display(by_category.round(2))

monthly_category_pivot = pd.pivot_table(
    orders,
    values="amount",
    index="month",
    columns="category",
    aggfunc="sum",
    fill_value=0
)
display(monthly_category_pivot.round(2))


In [ ]:
by_category["total_revenue"].plot(kind="bar", figsize=(9, 5))
plt.title("Total Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 6))
sns.heatmap(monthly_category_pivot, cmap="Blues", annot=False)
plt.title("Monthly Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Month")
plt.tight_layout()
plt.show()


## 5. Best customers and Pareto-style 80/20 analysis

In [ ]:
spend = orders.groupby("customer_id")["amount"].sum().sort_values(ascending=False)
top_n = max(1, int(np.ceil(len(spend) * 0.10)))
top_customers = spend.head(top_n)
top_share = top_customers.sum() / spend.sum()

print("Unique customers:", len(spend))
print("Top 10% customer count:", top_n)
print(f"Top 10% revenue share: {top_share:.1%}")
print("\\nTop 10 customers:")
display(spend.head(10).to_frame("total_spend").round(2))


In [ ]:
customer_cumulative_revenue = spend.cumsum() / spend.sum()
customer_cumulative_pct = np.arange(1, len(spend) + 1) / len(spend) * 100

plt.figure(figsize=(8, 5))
plt.plot(customer_cumulative_pct, customer_cumulative_revenue * 100)
plt.axvline(10, linestyle="--", label=f"Top 10% = {top_share:.1%} of revenue")
plt.xlabel("Cumulative share of customers (%)")
plt.ylabel("Cumulative share of revenue (%)")
plt.title("Customer Revenue Concentration")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Key findings and business recommendations

In [ ]:
total_revenue = orders["amount"].sum()
avg_order_value = orders["amount"].mean()
best_category = by_category.index[0]
best_category_revenue = by_category.iloc[0]["total_revenue"]
best_category_share = best_category_revenue / total_revenue

print(f"Total revenue: {total_revenue:,.2f}")
print(f"Average order value: {avg_order_value:,.2f}")
print(f"Best revenue month: {best_month} ({monthly.max():,.2f})")
print(f"Lowest revenue month: {worst_month} ({monthly.min():,.2f})")
print(f"Top category: {best_category} ({best_category_share:.1%} of revenue)")
print(f"Top 10% of customers contribute {top_share:.1%} of revenue")


### Recommendations
Use the computed values above to support these actions:

1. **Customer retention:** Create a loyalty or VIP programme for the top 10% of customers. Their measured revenue share is calculated in the preceding cell; monitor repeat purchase rate and retention after launch.
2. **Category planning:** Prioritise inventory and marketing tests for the highest-revenue category shown in the category summary. Compare margin and stock availability before increasing spend.
3. **Monthly campaign planning:** Review the best and weakest revenue months and month-over-month changes. Plan campaigns and inventory ahead of historically weaker months, then measure uplift against the baseline.

## Conclusion
This analysis demonstrates date feature engineering, monthly trend analysis, groupby summaries, pivot tables, and customer revenue concentration. Because the dataset is simulated, the findings are a practice exercise and should not be presented as actual company performance.


## 7. Export files
The following cell saves the cleaned data, summaries, pivot table, and dashboard to the current notebook folder.


In [ ]:
orders.to_csv("sample_orders.csv", index=False)
monthly_summary.to_csv("monthly_revenue.csv")
by_category.to_csv("category_summary.csv")
monthly_category_pivot.to_csv("monthly_category_pivot.csv")
spend.to_csv("customer_spend.csv")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].plot(monthly.index.astype(str), monthly.values, marker="o")
axes[0, 0].set(title="Monthly Revenue Trend", xlabel="Month", ylabel="Revenue")
axes[0, 0].tick_params(axis="x", rotation=45)

axes[0, 1].bar(by_category.index, by_category["total_revenue"])
axes[0, 1].set(title="Revenue by Category", xlabel="Category", ylabel="Revenue")
axes[0, 1].tick_params(axis="x", rotation=20)

axes[1, 0].bar(monthly_growth.index.astype(str), monthly_growth.fillna(0))
axes[1, 0].axhline(0, linewidth=1)
axes[1, 0].set(title="Month-over-Month Growth", xlabel="Month", ylabel="Growth (%)")
axes[1, 0].tick_params(axis="x", rotation=45)

axes[1, 1].plot(customer_cumulative_pct, customer_cumulative_revenue * 100)
axes[1, 1].axvline(10, linestyle="--", label=f"Top 10% = {top_share:.1%}")
axes[1, 1].set(title="Customer Revenue Concentration",
               xlabel="Cumulative customers (%)", ylabel="Cumulative revenue (%)")
axes[1, 1].legend()

fig.suptitle("E-commerce Sales Trend & Customer Analysis", fontsize=16)
plt.tight_layout()
plt.savefig("sales_analysis_dashboard.png", dpi=160, bbox_inches="tight")
plt.show()
